# 3 · Inference: score every customer and build the call list

| [1. Ingest](1_ingest.ipynb) | [2. Training](2_training.ipynb) | **3. Inference** (this notebook) |
| --- | --- | --- |
| Load the tables and build the index | Learn from past outcomes and save the model | Score every customer and build the call list |

**The business question** for all three notebooks: *which customers will place an order in the next 30 days?* The answer is a ranked call list the account team can work from.

**This notebook in one sentence:** load the saved model (no training), let it read each customer's history up to 15 November 2026, and estimate each customer's chance of ordering in the next 30 days.

| | |
| --- | --- |
| **Goes in** | The saved model from notebook 2 and the current data |
| **Comes out** | `output/call_list.csv`: all 500 customers, ranked |
| **In RAG terms** | Answering the questions: for each customer, retrieve their records, let the model answer, and show the sources |
| **Run time** | A few seconds |

In [1]:
import shop_demo as shop
from kumo_bundle import load_bundle, describe   # save and reload helpers, see kumo_bundle.py

shop.start()

## Step 1 · Open the data and the saved model

In production this would be last night's refreshed data; here it is the same files.
Before loading, `load_bundle` checks that the data is laid out the way the model was trained on: the same tables, columns, types and links.
A changed export then fails with a clear message instead of producing wrong scores quietly.

In [2]:
graph = shop.build_graph()
snapshot = graph.snapshot()
model = load_bundle(shop.MODEL_DIR, snapshot=snapshot, query=shop.QUESTION)
print("\n".join(describe(shop.MODEL_DIR).splitlines()[:6]))

Model bundle: models/orders-next-30d
Question (PQL): PREDICT COUNT(orders.*, 0, 30, days) > 0 FOR EACH customers.customer_id
Trained: 2026-11-24 07:46 UTC with kumo-engine 1.0.0.dev0+5d17fb3 (torch 2.15.0, Python 3.14.6)
Task type: binary_classification
Scoring defaults: binary_classification_threshold 0.5
Test results: ap 0.372, auprc 0.368, auroc 0.808


## Step 2 · Score every customer

For each of the 500 customers, Kumo retrieves their records up to 15 November 2026, and the model turns them into a probability.

In [3]:
scores = shop.score_everyone(model, snapshot)
print(f"Scored {len(scores)} customers as of {shop.AS_OF.date()}")

Scored 500 customers as of 2026-11-15


## Step 3 · The headline

Adding up every customer's probability gives the model's estimate of how many customers will order this month.
Last month's actual count, from the order file, is there for comparison.
Before relying on the estimate, check the Forecast row of notebook 2's exam: it shows how far the same sum was from what actually happened on dates where the answer is known.

In [4]:
shop.headline(scores)

Expected to order between 2026-11-15 and 2026-12-15: about 48 of 500 customers (the sum of every customer's probability)
For comparison, 36 customers ordered in the 30 days up to 2026-11-15 (from the order file)
Customers with a better than 50% chance: 20


## Step 4 · The call list

The chance of ordering comes from the model.
Name, region and order history are added from the customer and order files so the list can be read at a glance.

In [5]:
call_list = shop.call_list(scores)
call_list.head(15)

,customer_id,name,region,orders so far,last order,chance of ordering in next 30 days
rank,,,,,,
1,C120,Bright Biotech,South,28,2026-10-26,0.593
2,C345,Evergreen Insurance,West,47,2026-10-17,0.572
3,C428,Copper Foods,South,10,2026-11-12,0.547
4,C147,Blue Labs,West,14,2026-10-08,0.545
5,C080,Copper Health,South,26,2026-11-14,0.542
6,C448,Atlas Foods,Central,13,2026-11-08,0.538
7,C002,Globex,East,25,2026-10-15,0.537
8,C171,Lakeside Media,South,22,2026-11-13,0.537
9,C212,Bright Clinics,Central,17,2026-10-13,0.535


Read the list from the top: those are the accounts to contact first this month.
The accounts at the top have long order histories and recent orders, the recency-and-frequency pattern an experienced account manager looks for. Nobody wrote that rule: the model learned it from the practice questions.
Use the ranking rather than the 50% line. Few customers order in any given month, so only the strongest cases cross it.

## Step 5 · Why is a customer on the list? Show the sources

As a RAG answer cites its sources, Kumo can show every record it retrieved for one customer, which is everything the model read to score them.
First the top-ranked customer, then Acme.

In [6]:
shop.sources(model, snapshot, call_list.customer_id.iloc[0])

Bright Biotech (C120) as of 2026-11-15: 59% chance of ordering in the next 30 days, from 16 records


what the model saw
record   key                                                        
customer C120                   region South, signup_date 2023-02-10
order    900                       amount 600, order_date 2026-08-14
         1007                    amount 1,690, order_date 2026-10-09
         1032                      amount 230, order_date 2026-10-26
         1009                      amount 410, order_date 2026-10-10
         946                       amount 370, order_date 2026-09-09
         963                       amount 860, order_date 2026-09-16
         980                       amount 190, order_date 2026-09-25
         958                       amount 470, order_date 2026-09-14
         910                       amount 260, order_date 2026-08-19
         939                       amount 400, order_date 2026-09-03
product  P4    name Evergreen//One, category Subscription, price 500
         P2       name Support renewal, category Services, price 300
         P9         name FlashBlade//S, category Hardware, price 900
         P1       name FlashArray//X, category Hardware, price 1,200
         P7     name Cloud Block Store, category Software, price 400

In [7]:
shop.sources(model, snapshot, "C001")

Acme (C001) as of 2026-11-15: 29% chance of ordering in the next 30 days, from 6 records


what the model saw
record   key                                                     
customer C001                 region West, signup_date 2022-05-08
order    1017                   amount 300, order_date 2026-10-14
         988                    amount 450, order_date 2026-09-30
         1043                   amount 900, order_date 2026-11-02
product  P9      name FlashBlade//S, category Hardware, price 900
         P2    name Support renewal, category Services, price 300

Each score rests on a handful of records out of the 1,573 in the index: the customer's own row, their 10 most recent orders at most, and the products in those orders.

## Step 6 · Save the call list

In [8]:
call_list.to_csv(shop.OUTPUT_DIR / "call_list.csv")
print("Saved", shop.OUTPUT_DIR / "call_list.csv", f"({len(call_list)} customers)")

Saved output/call_list.csv (500 customers)


## In production

The same three steps, on a schedule:

| Step | Typical cadence | What changes from this demo |
| --- | --- | --- |
| 1. Ingest | Nightly | A warehouse connector (Snowflake, Databricks, BigQuery) replaces the files |
| 2. Training | Monthly, or when exam results drop | Each model is saved as a new version in object storage or a model registry |
| 3. Inference | Daily or weekly | The approved model scores the newest index, and the list goes to the CRM |